In [0]:
# Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Schema
TripSchema = StructType(
    [
        StructField("id", IntegerType()),
        StructField("client_id", IntegerType()),
        StructField("driver_id", IntegerType()),
        StructField("city_id", StringType()),
        StructField("status", StringType()),
        StructField("request_at", StringType()),
    ]
)

UserSchema = StructType(
    [
        StructField("user_id", IntegerType()),
        StructField("banned", StringType()),
        StructField("role", StringType()),
    ]
)

In [0]:
# Trip Data
TripData = [
    (1, 1, 10, 1, "completed", "2013-10-01"),
    (2, 2, 11, 1, "cancelled_by_driver", "2013-10-01"),
    (3, 3, 12, 6, "completed", "2013-10-01"),
    (4, 4, 13, 6, "cancelled_by_client", "2013-10-01"),
    (5, 1, 10, 1, "completed", "2013-10-02"),
    (6, 2, 11, 6, "completed", "2013-10-02"),
    (7, 3, 12, 6, "completed", "2013-10-02"),
    (8, 2, 12, 12, "completed", "2013-10-03"),
    (9, 3, 10, 12, "completed", "2013-10-03"),
    (10, 4, 13, 12, "cancelled_by_driver", "2013-10-03"),
]

tripDf = spark.createDataFrame(TripData, TripSchema)
tripDf.show()

In [0]:
# User Data

userData = [
    (1, "No", "client"),
    (2, "Yes", "client"),
    (3, "No", "client"),
    (4, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (12, "No", "driver"),
    (13, "No", "driver"),
]

userDf = spark.createDataFrame(userData, UserSchema)
userDf.show()

In [0]:
def cancellationRate(tripDf, userDf):
    # converting request_at string column to date type
    tripDf = tripDf.withColumn("request_at", col("request_at").cast("date"))

    # joining user and trip df to identify banned client
    clientBannedDf = (
        tripDf.alias("t1")
        .join(userDf.alias("t2"), col("t1.client_id") == col("t2.user_id"), how="left")
        .select("t1.*", col("t2.banned").alias("client_banned"))
    )

    # joining user and trip df to identify banned driver
    driverBannedDf = (
        clientBannedDf.alias("t1")
        .join(userDf.alias("t2"), col("t1.driver_id") == col("t2.user_id"), how="left")
        .select("t1.*", col("t2.banned").alias("driver_banned"))
    )

    # filtering out banned driver and client
    filteredDf = driverBannedDf.filter(
        (col("client_banned") != "Yes") & (col("driver_banned") != "Yes")
    )

    # calculating total rides each day
    totalRideDf = filteredDf.groupBy(col("request_at")).agg(
        count(col("id")).alias("total_rides")
    )

    # calculating cancelled rides each day
    cancelledRideDf = (
        filteredDf.filter(col("status").like("cancelled%"))
        .groupBy(col("request_at"))
        .agg(count(col("id")).alias("cancelled_rides"))
    )

    # joining both the calculated tables to calculate cancellation rate
    resultDf = (
        totalRideDf.alias("t1")
        .join(
            cancelledRideDf.alias("t2"),
            col("t1.request_at") == col("t2.request_at"),
            how="left",
        )
        .select(
            col("t1.request_at").alias("Day"),
            round(
                coalesce(col("t2.cancelled_rides"), lit(0)) / col("t1.total_rides"), 2
            ).alias("Cancellation Rate"),
        )
    )

    return resultDf

In [0]:
cancellationRate(tripDf, userDf).show()